# Chapter 9 — Ontologies and Natural Languages
### Notebook 0 · Overview and setup

*Book reference: Keet, *Ontology Engineering* (2nd ed.), Ch. 9*

An ontology nobody can read gets reviewed by nobody. Chapter 9 is about closing that gap in both directions — rendering axioms as readable sentences, and doing it in more than one language.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch09_toolkit as ch9
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

In [ ]:
import oe_course; print(json.dumps(oe_course.describe_environment(), indent=1))

## Notebooks in this chapter

| # | Notebook | Book section | What you build |
|---|---|---|---|
| 0 | `00_overview_and_setup` | — | environment check |
| 1 | `01_verbalisation` | 9.2 | a verbaliser **and its inverse** |
| 2 | `02_multilingual` | 9.1 | lexicons, coverage, and where translation leaks |
| 3 | `03_exercises` | 9.3 | autograded answers |
| 4 | `04_assignment` / `04_solutions` | — | problem set: a Claude verbaliser for multilingual review sheets, a validated judge, and an **optimal-stopping MDP** |


**By the end of this notebook you can:**

1. Verbalise axioms in controlled natural language, and **parse the sentences back** to check fidelity exactly.
2. Explain why a controlled language is the price of having an inverse.
3. Separate an ontology from its lexicons, and measure translation coverage per language.
4. State precisely what a round-trip check **cannot** see — and build the measurement that can.
5. Derive the stopping rule for best-of-n sampling.

## Why this chapter matters for evaluation

Almost everything in this course needs gold labels, a reasoner, or a judge. Verbalisation needs none of them: it has an **inverse**. Verbalise, parse back, compare — the grader is a function.

> That makes it the right place to be honest about the limits of exact metrics. `"No plant is a animal."` round-trips **perfectly** and is still wrong English. Notebook 1 shows exactly that, which is the most useful thing in the chapter.

### Sanity check: verbalise and recover every sample axiom

In [ ]:
results = [ch9.round_trips(a, 'en') for a in ch9.SAMPLE_AXIOMS]
print(f"{sum(r['ok'] for r in results)}/{len(results)} axioms survive the round trip")
assert all(r['ok'] for r in results)